# 03 — Matching Law

Fits the **Generalized Matching Law** (Baum, 1974):

$$\log\left(\frac{B_1}{B_2}\right) = a \cdot \log\left(\frac{R_1}{R_2}\right) + \log(b)$$

- $a$ = **sensitivity** (1.0 = perfect matching, <1 = undermatching, >1 = overmatching)
- $b$ = **bias** (1.0 = no bias, >1 = bias toward left/option 1)

Run `00_data_pipeline` first.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit

ROOT     = Path('..').resolve()
PROC_DIR = ROOT / 'data' / 'processed'
FIG_DIR  = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

def load(name):
    p = PROC_DIR / f'{name}.parquet'
    return pd.read_parquet(p) if p.exists() else pd.DataFrame()

# Scheduled reinforcement ratios per condition (from parameters.json)
SCHED_RATIOS = {
    'A': (3, 1),  # left:right
    'B': (2, 2),
    'C': (1, 3),
    'D': (5, 1),
    'E': (1, 5),
}


## 1. Compute B1, B2, R1, R2 per participant per session


In [ ]:
ml = load('matching_law')

if not ml.empty:
    # Condition is stored in the 'condition' column (A, B, C, D, E)
    choices = ml[ml['event_code'].isin(['LEFT', 'RIGHT', 'LEFT_SR', 'RIGHT_SR'])].copy()

    results = []
    for (pid, cond), grp in choices.groupby(['participant_id', 'condition']):
        B1 = (grp['event_code'] == 'LEFT').sum()
        B2 = (grp['event_code'] == 'RIGHT').sum()
        R1 = (grp['event_code'] == 'LEFT_SR').sum()
        R2 = (grp['event_code'] == 'RIGHT_SR').sum()

        # Scheduled ratio from parameters
        sched_r1, sched_r2 = SCHED_RATIOS.get(cond, (1, 1))

        total_B = B1 + B2
        total_R = R1 + R2

        results.append({
            'participant_id': pid,
            'condition':      cond,
            'B1': B1, 'B2': B2, 'R1': R1, 'R2': R2,
            'B1_prop': B1 / max(total_B, 1),
            'R1_prop': R1 / max(total_R, 1),
            'sched_R1': sched_r1, 'sched_R2': sched_r2,
            'sched_R1_prop': sched_r1 / (sched_r1 + sched_r2),
        })

    ml_df = pd.DataFrame(results)

    # Log ratios (exclude rows where B1==0, B2==0, R1==0, R2==0)
    valid = ml_df[(ml_df['B1'] > 0) & (ml_df['B2'] > 0) &
                  (ml_df['R1'] > 0) & (ml_df['R2'] > 0)].copy()
    valid['log_B_ratio'] = np.log(valid['B1'] / valid['B2'])
    valid['log_R_ratio'] = np.log(valid['R1'] / valid['R2'])
    valid['log_sched_ratio'] = np.log(valid['sched_R1'] / valid['sched_R2'])

    display(ml_df.round(3))


## 2. Fit the Generalized Matching Law


In [ ]:
if 'valid' in dir() and not valid.empty:
    gml_results = []

    for pid, pgrp in valid.groupby('participant_id'):
        if len(pgrp) < 3:
            print(f'{pid}: need ≥3 conditions for GML fit, skipping.')
            continue

        x = pgrp['log_R_ratio'].values
        y = pgrp['log_B_ratio'].values

        # OLS: y = a*x + log(b)
        slope, intercept, r, p_val, se = stats.linregress(x, y)
        a     = slope
        log_b = intercept
        b     = np.exp(log_b)

        gml_results.append({
            'participant_id': pid,
            'sensitivity_a':  round(a, 4),
            'bias_b':         round(b, 4),
            'log_b':          round(log_b, 4),
            'r_squared':      round(r**2, 4),
            'p_value':        round(p_val, 4),
            'n_conditions':   len(pgrp),
        })

    gml_df = pd.DataFrame(gml_results)
    print('=== GML Parameter Estimates ===')
    display(gml_df)
    print('\nInterpretation:')
    print('  a < 1.0  → undermatching (less extreme than matching predicts)')
    print('  a = 1.0  → strict matching')
    print('  a > 1.0  → overmatching')
    print('  b > 1.0  → bias toward left/option 1')
    print('  b < 1.0  → bias toward right/option 2')


## 3. Matching plot — obtained vs. predicted


In [ ]:
if 'valid' in dir() and 'gml_df' in dir() and not valid.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # --- Panel A: proportion matching ---
    ax = axes[0]
    for pid, pgrp in valid.groupby('participant_id'):
        ax.scatter(pgrp['R1_prop'], pgrp['B1_prop'], label=pid, s=60, alpha=0.8)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Strict matching')
    ax.set_xlabel('Proportion reinforcers from left (R1 / R1+R2)')
    ax.set_ylabel('Proportion choices to left (B1 / B1+B2)')
    ax.set_title('A — Obtained vs. Predicted Matching')
    ax.legend(fontsize=8)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)

    # --- Panel B: log ratio (GML) ---
    ax = axes[1]
    x_line = np.linspace(valid['log_R_ratio'].min() - 0.5,
                          valid['log_R_ratio'].max() + 0.5, 100)
    for pid, pgrp in valid.groupby('participant_id'):
        ax.scatter(pgrp['log_R_ratio'], pgrp['log_B_ratio'], label=pid, s=60, alpha=0.8)
        # Fitted line
        fit = gml_df[gml_df['participant_id'] == pid]
        if not fit.empty:
            a_fit = fit.iloc[0]['sensitivity_a']
            b_fit = fit.iloc[0]['log_b']
            ax.plot(x_line, a_fit * x_line + b_fit, '--', lw=1.5)

    ax.axline((0,0), slope=1, color='black', lw=1, ls=':', label='Strict matching (a=1)')
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_xlabel('log(R1 / R2)')
    ax.set_ylabel('log(B1 / B2)')
    ax.set_title('B — Generalized Matching Law Fit')
    ax.legend(fontsize=8)

    plt.tight_layout()
    fig.savefig(FIG_DIR / 'matching_law_gml.png', bbox_inches='tight')
    plt.show()


## 4. Sensitivity and bias summary


In [ ]:
if 'gml_df' in dir() and not gml_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].bar(gml_df['participant_id'], gml_df['sensitivity_a'],
                color='#457b9d', alpha=0.85)
    axes[0].axhline(1.0, ls='--', color='black', lw=1, label='Strict matching (a=1)')
    axes[0].set_ylabel('Sensitivity (a)')
    axes[0].set_title('Sensitivity')
    axes[0].legend(fontsize=8)

    axes[1].bar(gml_df['participant_id'], gml_df['bias_b'],
                color='#e63946', alpha=0.85)
    axes[1].axhline(1.0, ls='--', color='black', lw=1, label='No bias (b=1)')
    axes[1].set_ylabel('Bias (b)')
    axes[1].set_title('Bias')
    axes[1].legend(fontsize=8)

    plt.suptitle('GML Parameters by Participant', fontsize=12)
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'matching_law_parameters.png', bbox_inches='tight')
    plt.show()
